# Sentinel — research notebook
Walk through one issuer end to end: XBRL fundamentals → credit metrics → LLM debt-stack extraction → **deterministic verification** → recovery waterfall → memo.

Set `SEC_USER_AGENT` and, for the LLM steps, `SENTINEL_LLM_PROVIDER` plus an API key.

In [ ]:
import os, json, pandas as pd, matplotlib.pyplot as plt
os.chdir('..') if os.path.basename(os.getcwd())=='notebooks' else None
from sentinel.data import edgar
from sentinel.data.fundamentals import annual_metrics
from sentinel.analysis import credit
from sentinel.agents.extractor import extract_debt_stack
from sentinel.agents.verifier import verify_tranches
from sentinel.agents.writer import draft_memo
from sentinel.llm import get_provider
TICKER = 'CCL'

## 1. Fundamentals from XBRL

In [ ]:
cik = edgar.cik_for(TICKER)
years = annual_metrics(edgar.company_facts(cik))
hist = pd.DataFrame({fy: credit.credit_metrics(y) for fy, y in years.items()}).T.tail(6)
hist[['revenue','ebitda','debt','fcf']].div(1e9).round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
hist['net_leverage'].plot(ax=ax[0], marker='o', title='Net leverage (x)')
hist['interest_coverage'].plot(ax=ax[1], marker='o', title='Interest coverage (x)', color='C2')
plt.tight_layout()

## 2. Risk score and maturity wall

In [ ]:
fy = max(years); y = years[fy]; m = credit.credit_metrics(y)
runway = credit.liquidity_runway(y)
score, why = credit.risk_score(m, runway)
print(f'FY{fy} risk score {score}:', why)
pd.DataFrame(credit.maturity_wall(y), columns=['bucket','amount']).set_index('bucket').div(1e9).plot.bar(legend=False, title='Maturities ($bn)');

## 3. LLM extraction, then verification
The verifier is plain code, not another model. It checks each quote exists verbatim in the 10-K, that the amount is inside the quote, and that the total reconciles to XBRL debt.

In [ ]:
llm = get_provider()
tenk = next(f for f in edgar.recent_filings(cik) if f['form']=='10-K')
text = edgar.filing_text(tenk['url'])
tranches = extract_debt_stack(llm, edgar.debt_section(text))
v = verify_tranches(tranches, text, xbrl_total_debt=m['debt'])
pd.DataFrame(v['log'])

In [ ]:
print('precision:', v['precision'], '| reconciliation:', v['reconciliation'])

## 4. Recovery waterfall across EV scenarios

In [ ]:
if v['verified'] and m['ebitda'] > 0:
    wf = {k: credit.recovery_waterfall(ev, v['verified']) for k, ev in credit.ev_scenarios(m['ebitda']).items()}
    display(pd.DataFrame({k: {t['name']: round(t['recovery_pct'],2) for t in w} for k, w in wf.items()}))

## 5. Memo (numbers constrained to the fact sheet)

In [ ]:
memo = draft_memo(llm, {'ticker': TICKER, 'fiscal_year': fy, 'metrics': m, 'risk_score': score, 'risk_drivers': why})
print(json.dumps(memo, indent=1))

## 6. Where the AI fell short
Record failures here as you find them (quote mismatches, unit errors, missed tranches). This log is the evidence for the README's *Limitations* section.